# Ice–ocean flux consistency and the ocean + sea-ice salt budget

Water, salt and heat pass between the sea ice (CICE) and the ocean (MOM6) through the coupler (CMEPS). If nothing is lost on the way, what CICE says it sends to the ocean should equal what MOM6 says it receives. This notebook compares the two sides, globally integrated, month by month:

| Exchange | CICE sends (monthly mean) | MOM6 receives |
|---|---|---|
| Water | `fresh_ai_m` | `fsitherm` (daily mean, averaged to months) |
| Salt | `fsalt_ai_m` | `sfdsi` minus the restoring part `salt_flux_added` (monthly mean) |
| Heat | `fhocn_ai_m` | `total_seaice_melt_heat` (**daily snapshot**, averaged to months) |

The MOM6 heat term is only saved as one snapshot per day, so the heat comparison is a sampled estimate and will be noisier than water and salt.

It then closes the **ocean + sea-ice salt budget**: the only salt source for the combined system is surface salinity restoring, so ocean salt + ice salt should change only by the integrated restoring flux. A drift shows salt being created or lost at the ice–ocean interface (see [access-om3-configs#1580](https://github.com/ACCESS-NRI/access-om3-configs/issues/1580)).

Not repeated here:
- ocean + ice **water** conservation: [access-om3-configs#920](https://github.com/ACCESS-NRI/access-om3-configs/issues/920) and the water balance section of [`timeseries.ipynb`](timeseries.ipynb);
- MOM6-only heat, salt and mass closure: [`Closure_assessment.ipynb`](Closure_assessment.ipynb) ([#62](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/62)).

Issue: [access-om3-paper-1#206](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/206)

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
# e.g. https://access-om3-configs.access-hive.org.au/latest/Experiments/
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test5.1-8f50b769/datastore.json"

# papermill settings. *No need to modify these if running interactively.*
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory
nbname = None                          # notebook name

In [ ]:
if not papermill:
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import intake
import matplotlib.pyplot as plt
from distributed import Client

In [ ]:
client = Client(threads_per_worker=1)
print(client.dashboard_link)

### Open the intake-esm datastore

In [ ]:
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)

### The variables used

In [ ]:
def available_variables(datastore):
    '''Return a pandas dataframe summarising the variables in a datastore'''
    variable_columns = [col for col in datastore.df.columns if "variable" in col]
    return (
        datastore.df[variable_columns]
        .explode(variable_columns)
        .drop_duplicates()
        .set_index("variable")
        .sort_index()
    )

In [ ]:
cice_variables = ["fresh_ai_m", "fsalt_ai_m", "fhocn_ai_m", "hi_m", "sice_m"]
mom6_variables = ["fsitherm", "sfdsi", "salt_flux_added", "total_seaice_melt_heat", "masso", "soga"]

used = available_variables(datastore.search(variable=cice_variables + mom6_variables))
used = used.loc[cice_variables + mom6_variables]
used[["variable_long_name", "variable_units"]]

### Constants

CICE reports water fluxes in cm of fresh water per day. 1 cm of fresh water over 1 m² weighs 10 kg, so multiplying by 10/86400 gives kg m⁻² s⁻¹.

`RHO_ICE` is the CICE sea-ice density (CICE default, not overridden in `ice_in` for this run). Snow holds no salt, so snow density isn't needed for the salt budget.

In [ ]:
CM_PER_DAY_TO_KG_M2_S = 10.0 / 86400.0   # cm/day of fresh water -> kg m-2 s-1
RHO_ICE = 917.0                          # kg m-3, CICE sea-ice density (rhoi)
SA_OVER_SP = 35.16504 / 35.0             # MOM6 carries absolute salinity (USE_CONTEMP_ABSSAL = True) but soga is practical

### Cell areas for each model

In [ ]:
# CICE T-cell area from the CICE static grid (same as in SeaIce_area.ipynb)
tarea = datastore.search(variable="tarea", frequency="fx", realm="seaIce", path=".*output000.*").to_dask()["tarea"].compute()

# MOM6 cell area from the static file.
# path is a workaround - see https://forum.access-hive.org.au/t/issues-in-loading-ht-in-latest-conds-envs/6278
path = datastore.search(variable="areacello", filename="access-om3.mom6.static.nc").df.loc[0, "path"]
areacello = datastore.search(variable="areacello", path=path).to_dask()["areacello"].fillna(0.0).compute()

print(f"CICE total area (tarea):     {float(tarea.sum()):.6e} m2  (all cells, including land)")
print(f"MOM6 ocean area (areacello): {float(areacello.sum()):.6e} m2")

### A common monthly time axis

CICE stamps monthly means mid-month in a proleptic Gregorian calendar, MOM6 mid-month in a Gregorian calendar. To compare them exactly, each series is relabelled by the first day of its month.

In [ ]:
def label_by_month(da):
    '''Relabel the time axis of a monthly series with the first day of each month.'''
    month_start = pd.DatetimeIndex(da.time.dt.strftime("%Y-%m-01").values)
    return da.assign_coords(time=month_start)

### What CICE sends to the ocean

Monthly-mean CICE fluxes, multiplied by the cell area and summed over the globe. The `_ai` fluxes are already grid-cell means (weighted by ice concentration), so no extra factor of ice fraction is needed.

In [ ]:
cice = datastore.search(variable=cice_variables, frequency="1mon", realm="seaIce").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"time": 1, "nj": -1, "ni": -1},
        decode_timedelta=True
    )
)[cice_variables]

cice_sends = xr.Dataset({
    "water": (cice["fresh_ai_m"] * CM_PER_DAY_TO_KG_M2_S * tarea).sum(["nj", "ni"]),   # kg s-1
    "salt": (cice["fsalt_ai_m"] * tarea).sum(["nj", "ni"]),                            # kg s-1
    "heat": (cice["fhocn_ai_m"] * tarea).sum(["nj", "ni"]),                            # W
    # salt held in the sea ice: ice mass per area * bulk salinity (ppt -> kg/kg)
    "ice_salt": (RHO_ICE * cice["hi_m"] * 1e-3 * cice["sice_m"] * tarea).sum(["nj", "ni"]),  # kg
}).compute()

cice_sends = label_by_month(cice_sends)
cice_sends

### What MOM6 receives from the sea ice

- **Water:** `fsitherm` is saved as daily means. Each day is summed over the globe, then the days are averaged into months.
- **Salt:** MOM6 saves its net surface salt flux under the name `sfdsi`; it includes the restoring flux (`salt_flux_added`). The part from the sea ice is `sfdsi - salt_flux_added`. Both are monthly means.
- **Heat:** `total_seaice_melt_heat` is the global heat flux from sea ice into the ocean, saved as one **snapshot** per day. Averaging the daily snapshots into months gives a sampled monthly mean.

In [ ]:
# Water: daily-mean fsitherm, summed over the globe each day (kg s-1), then monthly means.
# Chunks of 30 days and the full horizontal grid keep the dask graph small for ~24,000 daily fields.
fsitherm = datastore.search(variable="fsitherm", frequency="1day").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"time": 30, "yh": -1, "xh": -1},
        decode_timedelta=True
    )
)["fsitherm"]

water_daily = (fsitherm.fillna(0.0) * areacello).sum(["yh", "xh"]).compute()
mom6_water = water_daily.resample(time="MS").mean()

In [ ]:
# Salt: monthly-mean net salt flux and its restoring part, summed over the globe (kg s-1)
salt_fluxes = {}
for variable in ["sfdsi", "salt_flux_added"]:
    flux = datastore.search(variable=variable, frequency="1mon").to_dask(
        xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
            compat="override",
            data_vars="minimal",
            coords="minimal",
        ),
        xarray_open_kwargs = dict(
            chunks={"time": 1, "yh": -1, "xh": -1},
            decode_timedelta=True
        )
    )[variable]
    salt_fluxes[variable] = (flux.fillna(0.0) * areacello).sum(["yh", "xh"])

salt_fluxes = label_by_month(xr.Dataset(salt_fluxes).compute())
mom6_salt_from_ice = salt_fluxes["sfdsi"] - salt_fluxes["salt_flux_added"]

In [ ]:
# Heat (and the scalars for the salt budget below): daily snapshots of global totals
scalars = datastore.search(variable=["total_seaice_melt_heat", "masso", "soga"], frequency="1day").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
)[["total_seaice_melt_heat", "masso", "soga"]].squeeze(drop=True).compute()

mom6_heat = scalars["total_seaice_melt_heat"].resample(time="MS").mean()   # W, sampled monthly mean

### CICE sends vs MOM6 receives

Months present on both sides are compared. The difference is what goes missing (or appears) between the two models.

In [ ]:
exchange = xr.Dataset({
    "water_cice": cice_sends["water"], "water_mom6": mom6_water,
    "salt_cice": cice_sends["salt"], "salt_mom6": mom6_salt_from_ice,
    "heat_cice": cice_sends["heat"], "heat_mom6": mom6_heat,
}).dropna("time")

print(f"{exchange.time.size} months on both sides: "
      f"{exchange.time[0].dt.strftime('%Y-%m').item()} to {exchange.time[-1].dt.strftime('%Y-%m').item()}")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 11), sharex=True)

panels = [
    ("water", 1e6, "Water (10$^6$ kg s$^{-1}$, i.e. Sv × 10$^{-3}$)"),
    ("salt", 1e3, "Salt (10$^3$ kg s$^{-1}$)"),
    ("heat", 1e12, "Heat (TW)"),
]

for row, (name, scale, label) in enumerate(panels):
    ax = axes[row, 0]
    (exchange[f"{name}_cice"] / scale).plot(ax=ax, label="CICE sends", lw=2)
    (exchange[f"{name}_mom6"] / scale).plot(ax=ax, label="MOM6 receives", ls="--")
    ax.set_ylabel(label)
    ax.set_title(f"{name.capitalize()}: ice to ocean, global")
    ax.set_xlabel("")
    ax.legend()

    ax = axes[row, 1]
    ((exchange[f"{name}_mom6"] - exchange[f"{name}_cice"]) / scale).plot(ax=ax, color="k")
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_ylabel(label)
    ax.set_title(f"{name.capitalize()}: MOM6 receives minus CICE sends")
    ax.set_xlabel("")

fig.tight_layout()

In [ ]:
# mkmd.savefig(fig, title, caption, dpi=100)
# Save figure and append to markdown summary.
mkmd.savefig(
    fig,
    "Ice to ocean fluxes: CICE vs MOM6",
    "Globally integrated monthly fluxes from sea ice to ocean as sent by CICE (fresh_ai_m, fsalt_ai_m, "
    "fhocn_ai_m) and as received by MOM6 (fsitherm; sfdsi - salt_flux_added; total_seaice_melt_heat, "
    "the last from daily snapshots). Right: MOM6 minus CICE.",
    dpi=150,
)

In [ ]:
rows = []
units = {"water": "kg s-1", "salt": "kg s-1", "heat": "W"}
for name in ["water", "salt", "heat"]:
    cice_side = exchange[f"{name}_cice"]
    difference = exchange[f"{name}_mom6"] - cice_side
    rows.append({
        "exchange": name,
        "mean |CICE flux|": f"{float(np.abs(cice_side).mean()):.3e} {units[name]}",
        "mean |MOM6 - CICE|": f"{float(np.abs(difference).mean()):.3e} {units[name]}",
        "ratio": f"{float(np.abs(difference).mean() / np.abs(cice_side).mean()):.1e}",
        "mean MOM6 - CICE (bias)": f"{float(difference.mean()):.3e} {units[name]}",
        "correlation": f"{float(xr.corr(cice_side, exchange[f'{name}_mom6'])):.6f}",
    })

exchange_table = pd.DataFrame(rows).set_index("exchange")
exchange_table

In [ ]:
# mkmd.table(title, table)
# Append table to markdown summary.
mkmd.table("Ice to ocean fluxes: CICE vs MOM6", exchange_table.to_markdown().split("\n"))

### Does `sfdsi` include salinity restoring?

MOM6 saves its net surface salt flux (`salt_flux`) under the CMOR name `sfdsi`, despite the long name "Downward Sea Ice Basal Salt Flux". If `sfdsi` includes restoring, then `sfdsi` minus `salt_flux_added` should match CICE better than `sfdsi` alone. In this configuration the restoring flux is adjusted to zero globally each step, so its global total is tiny and both versions agree to within that.

In [ ]:
print("Global mean |salt_flux_added| (restoring):  %.3e kg s-1" % float(np.abs(salt_fluxes["salt_flux_added"]).mean()))
print("Global mean |CICE fsalt_ai - sfdsi|:         %.3e kg s-1" % float(np.abs(cice_sends["salt"] - salt_fluxes["sfdsi"]).mean()))
print("Global mean |CICE fsalt_ai - (sfdsi - restoring)|: %.3e kg s-1" % float(np.abs(cice_sends["salt"] - mom6_salt_from_ice).mean()))

### Ocean + sea-ice salt budget

Total salt in the combined system = ocean salt + sea-ice salt (snow holds none):
- ocean salt = (35.16504/35) × `soga` × `masso` / 1000, from the daily snapshots, averaged into monthly means. The model carries absolute salinity (`USE_CONTEMP_ABSSAL = True`) but `soga` is written as practical salinity, hence the factor (see `Closure_assessment.ipynb`);
- ice salt = ρ_ice × `hi_m` × `sice_m` / 1000 × cell area, from CICE monthly means.

The only external salt source is restoring (`salt_flux_added`), so the change in total salt should equal its time integral. Monthly means of the products are used throughout, so short-term wiggles of order the month-to-month change are expected; a steady drift is not.

In [ ]:
ocean_salt = (1e-3 * SA_OVER_SP * scalars["soga"] * scalars["masso"]).resample(time="MS").mean()   # kg
ice_salt = cice_sends["ice_salt"]                                                      # kg

seconds_in_month = salt_fluxes.time.dt.days_in_month * 24 * 3600
restoring_added = (salt_fluxes["salt_flux_added"] * seconds_in_month).cumsum("time")  # kg, cumulative

salt_budget = xr.Dataset({
    "ocean": ocean_salt, "ice": ice_salt, "restoring": restoring_added,
}).dropna("time")
salt_budget["total"] = salt_budget["ocean"] + salt_budget["ice"]

# Changes since the first common month
salt_anomaly = salt_budget - salt_budget.isel(time=0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

ax = axes[0]
(salt_anomaly["ocean"] / 1e12).plot(ax=ax, label="ocean")
(salt_anomaly["ice"] / 1e12).plot(ax=ax, label="sea ice")
(salt_anomaly["total"] / 1e12).plot(ax=ax, label="ocean + sea ice", color="k", lw=2)
ax.set_ylabel("Change in salt since start (Tg)")
ax.set_title("Salt in the ocean and sea ice")
ax.set_xlabel("")
ax.legend()

ax = axes[1]
((salt_anomaly["total"] - salt_anomaly["restoring"]) / 1e12).plot(ax=ax, color="k")
ax.axhline(0, color="grey", lw=0.5)
ax.set_ylabel("Tg")
ax.set_title("Ocean + ice salt change minus integrated restoring")
ax.set_xlabel("")

fig.tight_layout()

In [ ]:
# mkmd.savefig(fig, title, caption, dpi=100)
# Save figure and append to markdown summary.
mkmd.savefig(
    fig,
    "Ocean + sea-ice salt budget",
    "Left: change in global salt content of the ocean (soga * masso / 1000), the sea ice "
    "(rho_ice * hi_m * sice_m / 1000) and their total. Right: total change minus the time-integrated "
    "restoring salt flux; a drift indicates salt created or lost at the ice-ocean interface.",
    dpi=150,
)

In [ ]:
years = (salt_budget.time[-1] - salt_budget.time[0]).dt.days.item() / 365.25
leak = float((salt_anomaly["total"] - salt_anomaly["restoring"]).isel(time=-1))
total_salt = float(salt_budget["total"].isel(time=0))
print(f"Over {years:.1f} years: ocean + ice salt minus restoring changed by {leak:.3e} kg "
      f"({leak / total_salt:.2e} of the total salt, {leak / years:.3e} kg per year)")

In [ ]:
client.close()